In [ ]:
import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import beta, binom, gamma, laplace, multinomial, norm, poisson


df = pd.read_csv(io.StringIO('thyroid_cancer_risk_data.csv'))


# ==============================================================================
# 1. DIRECT FITTING FUNCTIONS FOR ALL 7 DISTRIBUTIONS
# ==============================================================================

# 1. Gaussian direct fit
def direct_fit_gaussian(data, ax):
    mu, sigma = norm.fit(data)
    kc = np.linspace(mu - 3.5 * sigma, mu + 3.5 * sigma, 400)
    pdf = norm.pdf(kc, loc=mu, scale=sigma)
    ax.hist(data, bins="auto", density=True, alpha=0.5, color="skyblue", edgecolor="black")
    ax.plot(kc, pdf, "r-", lw=2.5, label=f"norm.fit (μ={mu:.2f}, σ={sigma:.2f})")
    ax.legend()

# 2. Laplace direct fit
def direct_fit_laplace(data, ax):
    loc, scale = laplace.fit(data)
    kc = np.linspace(loc - 4.0 * scale, loc + 4.0 * scale, 400)
    pdf = laplace.pdf(kc, loc=loc, scale=scale)
    ax.hist(data, bins="auto", density=True, alpha=0.5, color="coral", edgecolor="black")
    ax.plot(kc, pdf, "r-", lw=2.5, label=f"laplace.fit (μ={loc:.2f}, b={scale:.2f})")
    ax.legend()

# 3. Gamma direct fit
def direct_fit_gamma(data, ax):
    # Fix loc=0 to keep support on [0, inf)
    a, loc, scale = gamma.fit(data, floc=0)
    kc = np.linspace(1e-4, np.max(data) + 2.0, 400)
    pdf = gamma.pdf(kc, a=a, loc=loc, scale=scale)
    ax.hist(data, bins="auto", density=True, alpha=0.5, color="lightgreen", edgecolor="black")
    ax.plot(kc, pdf, "r-", lw=2.5, label=f"gamma.fit (a={a:.2f}, scale={scale:.2f})")
    ax.legend()

# 4. Beta direct fit
def direct_fit_beta(data, ax):
    # Fix bounds loc and scale to the data limits
    a_param, b_param, loc, scale = beta.fit(data, floc=np.min(data), fscale=np.ptp(data))
    kc = np.linspace(loc, loc + scale, 400)
    pdf = beta.pdf(kc, a=a_param, b=b_param, loc=loc, scale=scale)
    ax.hist(data, bins="auto", density=True, alpha=0.5, color="plum", edgecolor="black")
    ax.plot(kc, pdf, "r-", lw=2.5, label=f"beta.fit (α={a_param:.2f}, β={b_param:.2f})")
    ax.legend()

# 5. Poisson direct fit
def direct_fit_poisson(data, ax):
    lam = np.mean(data)
    max_k = int(np.max(data)) + 3
    kc = np.arange(0, max_k + 1)
    pmf = poisson.pmf(kc, mu=lam)
    ax.hist(data, bins=np.arange(-0.5, max_k + 1.5, 1), density=True, alpha=0.5, color="teal", edgecolor="black")
    ax.plot(kc, pmf, "ro-", lw=2.5, ms=6, label=f"poisson (λ={lam:.2f})")
    ax.set_xticks(kc)
    ax.legend()

# 6. Binomial direct fit
def direct_fit_binomial(data, n, ax):
    p = np.mean(data) / n
    kc = np.arange(0, n + 1)
    pmf = binom.pmf(kc, n=n, p=p)
    ax.hist(data, bins=np.arange(-0.5, n + 1.5, 1), density=True, alpha=0.5, color="mediumpurple", edgecolor="black")
    ax.plot(kc, pmf, "ro-", lw=2.5, ms=6, label=f"binom (n={n}, p={p:.2f})")
    ax.set_xticks(kc)
    ax.legend()

# 7. Multinomial direct fit
def direct_fit_multinomial(data, ax):
    categories = np.unique(data)
    K = len(categories)
    p_probs = np.array([np.mean(data == cat) for cat in categories])
    kc = np.arange(K)
    pmf = np.array([multinomial.pmf(np.eye(K, dtype=int)[i], n=1, p=p_probs) for i in range(K)])
    ax.bar(kc, pmf, alpha=0.5, color="goldenrod", edgecolor="black", width=0.4)
    ax.plot(kc, pmf, "ro-", lw=2.5, ms=7, label="multinomial.pmf")
    ax.set_xticks(kc)
    ax.set_xticklabels(categories)
    ax.legend()


# ==============================================================================
# 2. RUNNING DIRECT FITTING ON FEATURE SETS A, B, C, & D
# ==============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Feature Set A: T3_Level (Continuous -> Direct Beta Fit)
data_a = df["T3_Level"].values
direct_fit_beta(data_a, ax=axes[0, 0])
axes[0, 0].set_title("Feature Set A: T3_Level (Direct Beta Fit)")
axes[0, 0].set_xlabel("T3_Level")
axes[0, 0].set_ylabel("Probability Density")

# Feature Set B: Nodule_Size (Continuous -> Direct Beta Fit)
data_b = df["Nodule_Size"].values
direct_fit_beta(data_b, ax=axes[0, 1])
axes[0, 1].set_title("Feature Set B: Nodule_Size (Direct Beta Fit)")
axes[0, 1].set_xlabel("Nodule_Size")
axes[0, 1].set_ylabel("Probability Density")

# Feature Set C: Risk_Factor_Count (Discrete -> Direct Binomial Fit)
risk_flags = ["Radiation_Exposure", "Iodine_Deficiency", "Smoking", "Obesity"]
df["Risk_Factor_Count"] = (df[risk_flags] == "Yes").sum(axis=1)
data_c = df["Risk_Factor_Count"].values
direct_fit_binomial(data_c, n=4, ax=axes[1, 0])
axes[1, 0].set_title("Feature Set C: Risk_Factor_Count (Direct Binomial Fit n=4)")
axes[1, 0].set_xlabel("Risk Factor Count")
axes[1, 0].set_ylabel("Probability Mass")

# Feature Set D: Random Diagnostic Trials (Discrete -> Direct Binomial Fit)
np.random.seed(42)
N_trials, num_chunks = 30, 5000
p_malignant = (df["Diagnosis"] == "Malignant").mean()
trials = np.random.choice([1, 0], size=(num_chunks, N_trials), p=[p_malignant, 1.0 - p_malignant])
data_d = np.sum(trials, axis=1)

direct_fit_binomial(data_d, n=N_trials, ax=axes[1, 1])
axes[1, 1].set_title("Feature Set D: Random Diagnostic Trials (Direct Binomial Fit N=30)")
axes[1, 1].set_xlabel("Malignant Count per N=30 Chunk")
axes[1, 1].set_ylabel("Probability Mass")
axes[1, 1].set_xlim(np.min(data_d) - 1, np.max(data_d) + 1)

plt.tight_layout()
plt.show()